# Predicción de Precio de Autos Usados

Proyecto de regresión listo para **Google Colab**.

**Pregunta:** ¿podemos predecir el precio de un auto usado a partir de sus características?

**Target:** `price`

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset clásico de autos usados.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/CarPrice.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df.dtypes)
df.describe().T

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["price"], kde=True, bins=30)
plt.title("Distribución de precios")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="enginesize", y="price", ax=axes[0])
axes[0].set_title("Engine size vs Precio")
sns.scatterplot(data=df, x="horsepower", y="price", ax=axes[1])
axes[1].set_title("Horsepower vs Precio")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x="carbody", y="price")
plt.title("Tipo de carrocería vs Precio")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
num = df.select_dtypes(include=np.number)
sns.heatmap(num.corr(), annot=False, cmap="coolwarm", center=0)
plt.title("Correlación numérica")
plt.show()

### Hallazgos
- `enginesize`, `horsepower` y `curbweight` correlacionan fuerte con el precio.
- El tipo de carrocería y marca influyen.
- Dataset pequeño (~200 filas).

## 3. Preparar datos

In [ ]:
X = df.drop(columns=["price", "CarName"])
y = df["price"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred, squared=False),
        "R2": r2_score(y_test, y_pred),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("R2", ascending=False)

## 5. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)

plt.figure(figsize=(8, 5))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Top features — Random Forest")
plt.tight_layout()
plt.show()

## 6. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_trans = rf_pipe.named_steps["imputer"].transform(X_train)
X_train_trans = rf_pipe.named_steps["scaler"].transform(X_train_trans)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_trans)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values, X_train_trans, feature_names=X.columns, max_display=10, show=False)
plt.title("SHAP Summary — Autos usados")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_train_trans, feature_names=X.columns, plot_type="bar", max_display=10, show=False)
plt.title("Importancia media SHAP")
plt.tight_layout()
plt.show()

## 7. Conclusiones

- Variables de motor (`enginesize`, `horsepower`) son las más importantes.
- R² suele estar en **0.85–0.92** en test.
- Dataset pequeño: cuidado con overfitting.

**Rango esperado:** R² ~0.86–0.91 | MAE ~1500–2500